# 01 · CLIP 零样本分类与图文检索

**硬件**：🟢 CPU 可跑（ViT-B/32 约 600MB，首次运行需下载权重）

## 本 notebook 你将学到

1. CLIP 双塔结构如何把图像和文本编码到**同一个嵌入空间**
2. 零样本分类 = 图文相似度排序，没有任何分类头
3. Prompt 模板（`a photo of a {}`）为什么能显著影响效果
4. 亲眼看到嵌入空间里著名的 **modality gap** 现象
5. CLIP 的已知短板：计数

## 30 秒理论回顾（详见 [theory.md](../theory.md) 第 2 节）

CLIP 用 4 亿图文对训练：一个 batch 内，匹配的 (图, 文) 对相似度拉高，不匹配的拉低（InfoNCE 损失）。训练完成后，图像塔和文本塔的输出落在同一空间——于是任何分类任务都可以变成"这张图和哪句话最像"。

In [ ]:
%pip install -q torch transformers pillow matplotlib scikit-learn requests

In [ ]:
import torch
import matplotlib.pyplot as plt

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device = {device}")

## 1. 加载 CLIP

用最经典的 `openai/clip-vit-base-patch32`：图像塔是 ViT-B/32，文本塔是 12 层 Transformer。两塔输出各自过一个投影层后进入共享的 512 维空间。

In [ ]:
from transformers import CLIPModel, CLIPProcessor

MODEL_ID = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"参数量: {n_params:.0f}M，嵌入维度: {model.config.projection_dim}")

In [ ]:
# 下载几张 COCO 验证集图片作为实验素材（可以换成任何你自己的图片）
import requests
from io import BytesIO
from PIL import Image

IMAGE_URLS = [
    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "http://images.cocodataset.org/val2017/000000000285.jpg",
    "http://images.cocodataset.org/val2017/000000000139.jpg",
    "http://images.cocodataset.org/val2017/000000000785.jpg",
]

images = []
for url in IMAGE_URLS:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    images.append(Image.open(BytesIO(resp.content)).convert("RGB"))

fig, axes = plt.subplots(1, len(images), figsize=(16, 4))
for ax, img, url in zip(axes, images, IMAGE_URLS):
    ax.imshow(img)
    ax.set_title(url.split("/")[-1], fontsize=8)
    ax.axis("off")
plt.show()

## 2. 零样本分类

流程：把每个候选类别写成一句话 → 文本塔编码 → 与图像嵌入算余弦相似度 → 乘上温度系数（`logit_scale`，训练学出来的，约 100）→ softmax。

注意：**模型从未针对这些类别训练过分类器**，这就是"零样本"。

In [ ]:
labels = ["cat", "bear", "living room", "skier", "pizza", "motorcycle"]
texts = [f"a photo of a {l}" for l in labels]

inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    out = model(**inputs)

probs = out.logits_per_image.softmax(dim=-1).cpu()  # [n_images, n_labels]
print(f"logit_scale (温度) = {model.logit_scale.exp().item():.1f}\n")

fig, axes = plt.subplots(1, len(images), figsize=(16, 3))
for i, ax in enumerate(axes):
    ax.barh(labels, probs[i])
    ax.set_xlim(0, 1)
    ax.set_title(f"image {i}", fontsize=9)
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. 图文检索：相似度矩阵

检索就是分类的转置：给一句话，找最像的图。我们直接可视化整个 4×6 的余弦相似度矩阵——对角结构越明显，说明对齐质量越好。

In [ ]:
img_emb = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
txt_emb = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
sim = (img_emb @ txt_emb.T).cpu()

plt.figure(figsize=(7, 4))
plt.imshow(sim, cmap="viridis")
plt.colorbar(label="cosine similarity")
plt.xticks(range(len(labels)), labels, rotation=30)
plt.yticks(range(len(images)), [f"image {i}" for i in range(len(images))])
plt.title("Image-Text cosine similarity")
plt.show()

# 注意数值范围：CLIP 的余弦相似度通常挤在 0.1~0.35 之间，
# 排序有意义，但绝对值不是概率——这就是为什么需要温度系数放大后再 softmax。

## 4. Prompt 模板消融

CLIP 训练数据是网络 alt-text，多为完整短语而非孤立单词。所以 `"a photo of a cat"` 通常比裸词 `"cat"` 效果好。OpenAI 原论文用 80 个模板集成又能再涨几个点。

In [ ]:
def classify(img, texts):
    inp = processor(text=texts, images=img, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        return model(**inp).logits_per_image.softmax(dim=-1).cpu()[0]

templates = {
    "裸标签": "{}",
    "a photo of a {}": "a photo of a {}",
    "a blurry photo of a {}": "a blurry photo of a {}",
}

img = images[0]  # 用第一张图实验
for name, tpl in templates.items():
    p = classify(img, [tpl.format(l) for l in labels])
    top = p.argmax().item()
    print(f"{name:28s} -> top1: {labels[top]:12s} p={p[top]:.3f}")

## 5. 可视化嵌入空间：modality gap

把图像嵌入和文本嵌入一起降到 2 维。你会看到一个反直觉的现象：**图像点和文本点各占一边，中间有条沟**——即使是匹配的图文对也不重合。这就是 modality gap（Liang et al. 2022），源于初始化时两塔输出天然分离 + 对比损失只优化相对关系。

启示：跨模态比较只在"图 vs 文"的相对排序上有意义,不要拿绝对距离当语义距离。

In [ ]:
from sklearn.decomposition import PCA

all_emb = torch.cat([img_emb, txt_emb]).cpu().numpy()
xy = PCA(n_components=2).fit_transform(all_emb)
n = len(images)

plt.figure(figsize=(7, 5))
plt.scatter(xy[:n, 0], xy[:n, 1], c="tab:blue", label="image", s=80)
plt.scatter(xy[n:, 0], xy[n:, 1], c="tab:orange", label="text", s=80, marker="^")
for i, l in enumerate(labels):
    plt.annotate(l, xy[n + i], fontsize=8)
for i in range(n):
    plt.annotate(f"img{i}", xy[i], fontsize=8)
plt.legend()
plt.title("CLIP embedding space (PCA) — 注意图像簇与文本簇的分离")
plt.show()

## 6. 短板实验：计数

CLIP 的对比训练只需要"抓住主要语义"就能区分 batch 内负样本，所以数量、空间关系这类组合信息学得很差。第一张 COCO 图里有两只猫——看看 CLIP 知不知道。

In [ ]:
counting = [f"a photo of {n} cats" for n in ["one", "two", "three", "four"]]
p = classify(images[0], counting)
for t, prob in zip(counting, p):
    print(f"{t:28s} {prob:.3f}")
print("\n概率接近均匀分布 → CLIP 基本不会数数。")
print("这类短板正是 01 章 VLM（把视觉特征交给 LLM 推理）要解决的问题。")

## 练习

1. 把 `MODEL_ID` 换成 `google/siglip2-base-patch16-224`（注意改用 `AutoModel`/`AutoProcessor`，SigLIP 用 sigmoid 而非 softmax，接口略有不同），对比同一组任务的表现。
2. 用你自己的 20 张照片建一个小相册，实现"用一句话搜照片"。
3. 构造一对"空间关系"文本（`a cat on the left of a dog` / `a dog on the left of a cat`），验证 CLIP 是否能区分。

**下一站**：[02_tokenize_everything.ipynb](02_tokenize_everything.ipynb) — 看看各模态是怎么变成 token 的。